<a href="https://colab.research.google.com/github/nivethithanm/torchcode-solutions/blob/main/TC_06a_multihead_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FIX-07: Multi-Head Attention — From Single Head to Multi


**The core idea**: Multi-head attention is NOT a new algorithm. It's single-head attention run **in parallel** on smaller slices of the input, then concatenated.

Think of it like this:
```
Single-head:  Input (B, N, 512) → one attention on 512 dims → Output (B, N, 512)
Multi-head:   Input (B, N, 512) → 8 attentions on 64 dims each → concat → Output (B, N, 512)
```

Why? Each head can learn to attend to DIFFERENT things. Head 1 might attend to nearby words, Head 2 to the subject of the sentence, Head 3 to punctuation, etc.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

## Part 1: Multi-head as a loop (slow but crystal clear)

We'll literally create separate single-head attentions and loop over them.

In [2]:
class SingleHeadAttention(nn.Module):
    """You already know this one."""
    def __init__(self, d_in, d_head):
        super().__init__()
        self.W_q = nn.Linear(d_in, d_head, bias=False)
        self.W_k = nn.Linear(d_in, d_head, bias=False)
        self.W_v = nn.Linear(d_in, d_head, bias=False)
        self.scale = math.sqrt(d_head)

    def forward(self, x):
        Q = self.W_q(x)  # (B, N, d_head)
        K = self.W_k(x)  # (B, N, d_head)
        V = self.W_v(x)  # (B, N, d_head)
        scores = Q @ K.transpose(-2, -1) / self.scale  # (B, N, N)
        weights = F.softmax(scores, dim=-1)
        return weights @ V  # (B, N, d_head)


class NaiveMultiHead(nn.Module):
    """Multi-head = loop over independent single heads, then concat."""
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_head = d_model // n_heads
        # Create n_heads separate attention modules
        self.heads = nn.ModuleList([
            SingleHeadAttention(d_model, self.d_head)
            for _ in range(n_heads)
        ])
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        # Run each head independently
        head_outputs = [head(x) for head in self.heads]  # list of (B, N, d_head)

        # Concatenate along last dim
        concat = torch.cat(head_outputs, dim=-1)  # (B, N, d_model)

        # Final projection
        return self.W_o(concat)  # (B, N, d_model)

# Test
naive_mha = NaiveMultiHead(d_model=64, n_heads=4)
x = torch.randn(2, 10, 64)
out = naive_mha(x)
print(f'Input:  {x.shape}')  # (2, 10, 64)
print(f'Output: {out.shape}')  # (2, 10, 64) — same shape!
print(f'Heads:  4 × (B, N, 16) → cat → (B, N, 64)')
print('✅ Naive multi-head works! But the loop is slow...')

Input:  torch.Size([2, 10, 64])
Output: torch.Size([2, 10, 64])
Heads:  4 × (B, N, 16) → cat → (B, N, 64)
✅ Naive multi-head works! But the loop is slow...


## Part 2: The reshape trick — eliminate the loop

Instead of 4 separate W_q matrices of size (64, 16), use ONE big W_q of size (64, 64).
Then RESHAPE the output to create the heads.

This is the ONLY new concept. The attention computation is identical.

In [3]:
B, N, D = 2, 10, 64
n_heads = 4
d_head = D // n_heads  # 16

x = torch.randn(B, N, D)

# One big projection instead of 4 small ones
W_q = nn.Linear(D, D, bias=False)  # (64, 64) — NOT (64, 16)

# Project
Q = W_q(x)  # (2, 10, 64)
print(f'After projection: {Q.shape}')

# Now the KEY step: RESHAPE to create heads
# (B, N, D) → (B, N, n_heads, d_head) → (B, n_heads, N, d_head)

# Step 2a: split the last dimension into (n_heads, d_head)
Q_split = Q.view(B, N, n_heads, d_head)
print(f'After view:      {Q_split.shape}')  # (2, 10, 4, 16)

# Step 2b: move heads dimension before sequence dimension
Q_heads = Q_split.transpose(1, 2)
print(f'After transpose: {Q_heads.shape}')  # (2, 4, 10, 16)
# Now it's (Batch, Heads, Seq, HeadDim) — ready for attention!

After projection: torch.Size([2, 10, 64])
After view:      torch.Size([2, 10, 4, 16])
After transpose: torch.Size([2, 4, 10, 16])


In [4]:
# Let's see: Q_heads[b, h, :, :] is the query for batch b, head h
# It has shape (N, d_head) — EXACTLY like single-head attention!

# So Q @ K^T now operates on each (batch, head) independently:
K_heads = torch.randn(B, n_heads, N, d_head)  # pretend we did the same for K

scores = Q_heads @ K_heads.transpose(-2, -1)  # (B, H, N, d_head) @ (B, H, d_head, N)
print(f'Scores: {scores.shape}')  # (B, 4, 10, 10) — N×N scores per head!
print('Each head has its OWN attention pattern!')

Scores: torch.Size([2, 4, 10, 10])
Each head has its OWN attention pattern!


## Part 3: After attention — merge heads back

In [5]:
# After attention, we have output of shape (B, n_heads, N, d_head)
attn_output = torch.randn(B, n_heads, N, d_head)
print(f'Per-head output: {attn_output.shape}')  # (2, 4, 10, 16)

# Merge: reverse the split
# Step 1: move heads back next to d_head
merged = attn_output.transpose(1, 2)  # (B, N, n_heads, d_head)
print(f'After transpose: {merged.shape}')  # (2, 10, 4, 16)

# Step 2: flatten heads × d_head back into D
# IMPORTANT: .contiguous() is needed because transpose makes memory non-contiguous
merged = merged.contiguous().view(B, N, D)
print(f'After view:      {merged.shape}')  # (2, 10, 64) — back to original shape!

Per-head output: torch.Size([2, 4, 10, 16])
After transpose: torch.Size([2, 10, 4, 16])
After view:      torch.Size([2, 10, 64])


## Part 4: Put it all together

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = math.sqrt(self.d_head)

        # One big projection for each of Q, K, V
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def _split_heads(self, x):
        """(B, N, D) → (B, n_heads, N, d_head)"""
        B, N, D = x.shape
        return x.view(B, N, self.n_heads, self.d_head).transpose(1, 2)

    def _merge_heads(self, x):
        """(B, n_heads, N, d_head) → (B, N, D)"""
        B, H, N, d = x.shape
        return x.transpose(1, 2).contiguous().view(B, N, H * d)

    def forward(self, x, mask=None):
        # Step 1: Project
        Q = self._split_heads(self.W_q(x))  # (B, H, N, d_head)
        K = self._split_heads(self.W_k(x))  # (B, H, N, d_head)
        V = self._split_heads(self.W_v(x))  # (B, H, N, d_head)

        # Step 2: Attention (SAME as single-head, but batched over heads)
        scores = Q @ K.transpose(-2, -1) / self.scale  # (B, H, N, N)

        if mask is not None:
            scores = scores.masked_fill(~mask.unsqueeze(0).unsqueeze(0), float('-inf'))

        weights = F.softmax(scores, dim=-1)  # (B, H, N, N)
        attn_out = weights @ V  # (B, H, N, d_head)

        # Step 3: Merge heads and project
        merged = self._merge_heads(attn_out)  # (B, N, D)
        return self.W_o(merged)  # (B, N, D)

mha = MultiHeadAttention(d_model=64, n_heads=4)
x = torch.randn(2, 10, 64)
out = mha(x)
assert out.shape == (2, 10, 64)

# With causal mask
mask = torch.tril(torch.ones(10, 10, dtype=torch.bool))
out_causal = mha(x, mask=mask)
assert out_causal.shape == (2, 10, 64)
print('✅ Multi-Head Attention — now you see it\'s just single-head with a reshape!')

✅ Multi-Head Attention — now you see it's just single-head with a reshape!


## Exercise: Implement _split_heads and _merge_heads yourself

In [7]:
def split_heads(x, n_heads):
    """(B, N, D) → (B, n_heads, N, d_head)
    TODO: implement with view + transpose
    """
    B, N, D = x.shape
    d_head = D // n_heads
    # YOUR CODE: two operations
    split = x.view(B, N, n_heads, d_head)
    split = split.transpose(1, 2)
    return split

def merge_heads(x):
    """(B, n_heads, N, d_head) → (B, N, D)
    TODO: implement with transpose + contiguous + view
    """
    B, H, N, d = x.shape
    # YOUR CODE: three operations
    merge = x.transpose(1, 2).contiguous()
    merge = merge.view(B, N, H*d)
    return merge

x = torch.randn(2, 10, 64)
split = split_heads(x, 4)
assert split.shape == (2, 4, 10, 16)
back = merge_heads(split)
assert back.shape == (2, 10, 64)
assert torch.allclose(x, back)
print('✅ You own the reshape trick!')

✅ You own the reshape trick!
